In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from src.qkd import generate_bb84_job, calculate_qber, encrypt_message, decrypt_message
from processors.hardware import processor_a, processor_b
import matplotlib.pyplot as plt

# 1. Generate BB84 protocol circuits with enough bits for encryption
n_bits = 1500
circuits, alice_bits, alice_bases, bob_bases = generate_bb84_job(n_bits=n_bits)

def run_processor_experiment(processor, name):
    print(f"\n--- Running Secure Channel on {name} ---")
    backend = processor["backend"]
    noise_model = processor["noise_model"]
    
    sim = AerSimulator(noise_model=noise_model)
    
    bob_results = []
    total_depth = 0
    total_gates = 0
    
    for qc in circuits:
        transpiled_qc = transpile(qc, backend=backend, optimization_level=1)
        total_depth += transpiled_qc.depth()
        total_gates += transpiled_qc.size()
        
        job = sim.run(transpiled_qc, shots=1, memory=True)
        result = job.result()
        measured_bit = result.get_memory()[0]
        bob_results.append(measured_bit)
        
    qber, key_a, key_b = calculate_qber(alice_bits, alice_bases, bob_bases, bob_results)
    
    print(f"Average Circuit Depth: {total_depth / n_bits:.2f}")
    print(f"Average Gate Count: {total_gates / n_bits:.2f}")
    print(f"Calculated QBER: {qber * 100:.2f}%")
    print(f"Sifted Key Length: {len(key_a)} bits")
    
    return qber, key_a, key_b

# 2. Run simulation across both processors
qber_a, key_a_alice, key_a_bob = run_processor_experiment(processor_a, "Processor A")
qber_b, key_b_alice, key_b_bob = run_processor_experiment(processor_b, "Processor B")

# 3. Transmit an actual encrypted secret message using Processor A's quantum key
secret_message = "Hello Bob! This is a fully quantum-secure communication channel."
print(f"\nOriginal Secret Message: '{secret_message}'")

ciphertext, msg_len = encrypt_message(secret_message, key_a_alice)
received_message = decrypt_message(ciphertext, key_a_bob, msg_len)
print(f"Successfully Decrypted Message: '{received_message}'")

# 4. Generate and save the comparison plot
plt.figure(figsize=(6, 4))
plt.bar(["Processor A", "Processor B"], [qber_a * 100, qber_b * 100], color=['#2b5c8f', '#d9534f'])
plt.ylabel("QBER (%)")
plt.title("Quantum Secure Channel: Noise & QBER Comparison")
plt.savefig("results/figures/qber_comparison.png")
plt.show()


--- Running Secure Channel on Processor A ---
Average Circuit Depth: 2.78
Average Gate Count: 2.78
Calculated QBER: 0.00%
Sifted Key Length: 492 bits

--- Running Secure Channel on Processor B ---
Average Circuit Depth: 2.78
Average Gate Count: 2.78
Calculated QBER: 0.00%
Sifted Key Length: 492 bits

Original Secret Message: 'Hello Bob! This is a fully quantum-secure communication channel.'


ValueError: Sifted key is too short for this message! Increase n_bits.